In [ ]:
import pandas as pd
import numpy as np
%pip install holidays
%pip install tensorflow
%pip install keras_tuner
import holidays
import os

# --- NEW: Force TensorFlow to use CPU ---
# This MUST be before the tensorflow import
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Import TensorFlow, Keras, and KerasTuner
import tensorflow as tf
from tensorflow import keras
# --- NEW: Import regularizers ---
from tensorflow.keras import layers, regularizers
import keras_tuner as kt

print("A carregar os dados...")
dfTrain = pd.read_csv('training_data.csv', encoding='latin1')
dfTest = pd.read_csv('test_data.csv', encoding='latin1')


print("A executar a engenharia de features...")

# ### --- START: UNCHANGED FEATURE ENGINEERING --- ###
for df in [dfTrain, dfTest]:
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['dayOfWeek'] = df['record_date'].dt.dayofweek
    df['month'] = df['record_date'].dt.month
    df['year'] = df['record_date'].dt.year
    df['isRushHour'] = ((df['hour'] >= 7) & (df['hour'] <= 10) | 
                        (df['hour'] >= 16) & (df['hour'] <= 21)).astype(int)
    df['IS_WEEKEND'] = (df['dayOfWeek'] >= 5).astype(int)
    anos = df['year'].unique()
    feriados_portugal = holidays.Portugal(years=anos)
    df['is_holiday'] = df['record_date'].dt.date.isin(feriados_portugal).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['dayOfWeek_sin'] = np.sin(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['dayOfWeek_cos'] = np.cos(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['TIME_DELAY_RATIO'] = df['AVERAGE_TIME_DIFF'] / df['AVERAGE_FREE_FLOW_TIME']
    df['FREE_FLOW_RATIO'] = df['AVERAGE_FREE_FLOW_TIME'] / (df['AVERAGE_FREE_FLOW_TIME'] + df['AVERAGE_TIME_DIFF'])
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)


print("A mapear features categóricas...")

mappingLuminosity = {'DARK': 0, 'LOW_LIGHT': 1, 'LIGHT': 2}
mappingCloudiness = {'céu limpo': 0, 'céu claro': 0, 'céu pouco nublado': 1, 'algumas nuvens': 1, 'nuvens dispersas': 2, 'nuvens quebradas': 3, 'nuvens quebrados': 3, 'nublado': 4, 'tempo nublado': 4}
mappingRain = {'chuvisco fraco': 0, 'chuvisco e chuva fraca': 1, 'chuva fraca': 1, 'chuva leve': 1, 'aguaceiros fracos': 2, 'chuva': 2, 'aguaceiros': 3, 'chuva moderada': 3, 'chuva forte': 4, 'chuva de intensidade pesada': 5, 'chuva de intensidade pesado': 5, 'trovoada com chuva leve': 5, 'trovoada com chuva': 6}

for df in [dfTrain, dfTest]:
    df['LUMINOSITY'] = df['LUMINOSITY'].map(mappingLuminosity).fillna(-1)
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].map(mappingCloudiness).fillna(-1)
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].map(mappingRain).fillna(-1)


print("A preparar os dataframes X e y finais...")

mappingSpeedDiff = {'Low': 0, 'Medium': 1, 'High': 2, 'Very_High': 3}
dfTrain['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF'].map(mappingSpeedDiff).fillna(-1)


p1_inicio = pd.to_datetime('2018-09-12')
p1_fim = pd.to_datetime('2018-12-14')

p2_inicio = pd.to_datetime('2019-01-03')
p2_fim = pd.to_datetime('2019-04-05')

p3_inicio = pd.to_datetime('2019-04-23')
p3_fim = pd.to_datetime('2019-06-21')

cond_p1 = (dfTrain['record_date'] >= p1_inicio) & (dfTrain['record_date'] <= p1_fim)
cond_p2 = (dfTrain['record_date'] >= p2_inicio) & (dfTrain['record_date'] <= p2_fim)
cond_p3 = (dfTrain['record_date'] >= p3_inicio) & (dfTrain['record_date'] <= p3_fim)
cond_total = cond_p1 | cond_p2 | cond_p3
dfTrain['TEMPO_AULAS'] = (cond_total).astype(int)

cond_p1 = (dfTest['record_date'] >= p1_inicio) & (dfTest['record_date'] <= p1_fim)
cond_p2 = (dfTest['record_date'] >= p2_inicio) & (dfTest['record_date'] <= p2_fim)
cond_p3 = (dfTest['record_date'] >= p3_inicio) & (dfTest['record_date'] <= p3_fim)
cond_total = cond_p1 | cond_p2 | cond_p3
dfTest['TEMPO_AULAS'] = (cond_total).astype(int)

colunas_a_remover = [
    'AVERAGE_SPEED_DIFF',
    'AVERAGE_PRECIPITATION',
    'LUMINOSITY',
    'dayOfWeek',
    'record_date', 
    'city_name',
    'year',
    'hour'
    'month',
    'day',    
]
X = dfTrain.drop(columns=colunas_a_remover, errors='ignore')
y = dfTrain['AVERAGE_SPEED_DIFF']

X_teste = dfTest.drop(columns=[col for col in colunas_a_remover if col in dfTest.columns], errors='ignore')
X, X_teste = X.align(X_teste, join='inner', axis=1, fill_value=0)
# ### --- END: UNCHANGED FEATURE ENGINEERING --- ###


# ### --- NEW: PREPROCESSING FOR NEURAL NETWORK --- ###

print("\n(Novo) A escalar os dados para a Neural Network...")
# 1. Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_teste_scaled = scaler.transform(X_teste)
num_features = X_scaled.shape[1]

print("(Novo) A re-mapear a variável alvo...")
# 2. Remap Target Variable 'y'
y_remap = {-1: 0, 0: 1, 1: 2, 2: 3, 3: 4}
y_mapped = y.map(y_remap)
n_classes = len(y_remap)

# Define the reverse map for submission
reverse_map = {0: 'None', 1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very_High'}

# 3. Create Validation Split for Tuning
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_mapped, 
    test_size=0.20, 
    random_state=2020, 
    stratify=y_mapped
)

# 4. Calculate Class Weights
classes_train = np.unique(y_train)
weights_train = compute_class_weight('balanced', classes=classes_train, y=y_train)
class_weight_dict_train = dict(zip(classes_train, weights_train))

classes_full = np.unique(y_mapped)
weights_full = compute_class_weight('balanced', classes=classes_full, y=y_mapped)
class_weight_dict_full = dict(zip(classes_full, weights_full))

print(f"Features de entrada: {num_features}, Classes de saída: {n_classes}")


# ### --- NEW: NEURAL NETWORK MODEL DEFINITION & TUNING (WITH L2) --- ###

print("\n(Novo) A definir o modelo de Neural Network para tuning...")

def build_model(hp):
    # --- NEW: Tune the L2 penalty value ---
    hp_l2 = hp.Choice('l2_lambda', values=[1e-2, 1e-3, 1e-4, 1e-5])
    
    model = keras.Sequential()
    model.add(layers.Input(shape=(num_features,)))
    
    # Tune the number of hidden units
    hp_units = hp.Int('units', min_value=32, max_value=128, step=32)
    model.add(layers.Dense(
        units=hp_units, 
        activation='relu',
        # --- NEW: Add L2 regularizer ---
        kernel_regularizer=regularizers.l2(hp_l2)
    ))
    model.add(layers.BatchNormalization())
    
    # Tune the dropout rate
    hp_dropout = hp.Float('dropout', min_value=0.1, max_value=0.5, step=0.1)
    model.add(layers.Dropout(rate=hp_dropout))
    
    # Optionally tune a second layer
    if hp.Boolean('two_layers'):
        hp_units_2 = hp.Int('units_2', min_value=16, max_value=64, step=16)
        model.add(layers.Dense(
            units=hp_units_2, 
            activation='relu',
            # --- NEW: Add L2 regularizer here too ---
            kernel_regularizer=regularizers.l2(hp_l2)
        ))
        model.add(layers.BatchNormalization())
        hp_dropout_2 = hp.Float('dropout_2', min_value=0.1, max_value=0.3, step=0.1)
        model.add(layers.Dropout(rate=hp_dropout_2))

    # Output layer
    model.add(layers.Dense(n_classes, activation='softmax'))
    
    # Tune the learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# --- NEW: Setup a more thorough tuner ---
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=30,  # Increased from 10
    executions_per_trial=2, # Run each model 2x for stability
    directory='nn_tuner',
    project_name='traffic_speed_nn'
)

# --- NEW: Define all callbacks ---
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,   # Reduce LR by 80%
    patience=3,   # If no improvement for 3 epochs
    min_lr=1e-6,  # Minimum learning rate
    verbose=1
)

# Combine callbacks into a list
callbacks_list = [early_stopping, lr_scheduler]


print("\n(Novo) A iniciar a otimização de hiperparâmetros (KerasTuner)...")
tuner.search(
    X_train, y_train,
    epochs=40,
    validation_data=(X_val, y_val),
    # --- NEW: Use the full callbacks list ---
    callbacks=callbacks_list,
    class_weight=class_weight_dict_train,
    verbose=1
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Melhores Hiperparâmetros Encontrados:
Units: {best_hps.get('units')}
Dropout: {best_hps.get('dropout')}
L2 Lambda: {best_hps.get('l2_lambda')}
Two Layers: {best_hps.get('two_layers')}
Learning Rate: {best_hps.get('learning_rate')}
(E outros params se 'two_layers' for True)
""")


# ### --- NEW: FINAL MODEL TRAINING & SUBMISSION --- ###

print("\n(Novo) A treinar o modelo final com todos os dados de treino...")
# Build the final model with the best hyperparameters
final_model = tuner.hypermodel.build(best_hps)

# Train on the FULL training dataset
final_history = final_model.fit(
    X_scaled, y_mapped,
    epochs=200, # Train for more epochs on the final model
    batch_size=32,
    validation_split=0.1, # Use 10% of data for validation
    # --- NEW: Use the full callbacks list here too ---
    callbacks=callbacks_list,
    class_weight=class_weight_dict_full,
    verbose=1
)

# (Optional) Show performance on the validation set we created earlier
print("\n--- (Novo) PERFORMANCE NO CONJUNTO DE VALIDAÇÃO ---")
y_pred_probs_val = final_model.predict(X_val)
y_pred_val = np.argmax(y_pred_probs_val, axis=1) # Get the class with highest prob
print(f"Accuracy no conjunto de validação: {accuracy_score(y_val, y_pred_val):.4f}")
print(classification_report(y_val, y_pred_val, target_names=reverse_map.values()))


print("\n(Novo) A gerar o ficheiro de submissão...")
# Make predictions on the test set
final_predictions_probs = final_model.predict(X_teste_scaled)
final_predictions_numeric = np.argmax(final_predictions_probs, axis=1)

# Map numeric predictions back to string labels
final_predictions_labels = [reverse_map[pred] for pred in final_predictions_numeric]

# Create submission file (same as your original code)
row_ids = range(1, len(dfTest) + 1)
output = pd.DataFrame({'RowId': row_ids, 'Speed_Diff': final_predictions_labels})
output.to_csv('submission_nn.csv', index=False) # Saved as new file

print("\nFicheiro 'submission_nn.csv' criado!")
print("\nDistribuição das classes na submissão:")
print(output['Speed_Diff'].value_counts())